# Local Delivery And Migrations

## Setup
Use the separate engineering environment in README.md. Read the backend primer first. These assignments use local services and your own files; no model/API calls are made by default. All records and keys are invented teaching examples. Keep the whole track folder. Edit the numbered TODOs in `submission.py`, then run `importlib.reload(impl)` before rerunning tests. Instructor implementations are not included in the student handout.

Use a two-pass workflow: attempt each small TODO using the contract, then consult the staged hints in HINTS.md if needed. Record how much help you used in your learning log. Keep your solution in your own Git branch or copy. Never replace it with the instructor implementation before attempting the exercise.

In [1]:
from pathlib import Path
import sys, importlib, json
HERE = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "engineeringlab").is_dir() or (p / "labs/engineering/engineeringlab").is_dir())
TRACK = HERE if (HERE / "engineeringlab").is_dir() else HERE / "labs/engineering"
if str(TRACK) not in sys.path: sys.path.insert(0, str(TRACK))
RUN_EXERCISES = False
MODULE = "submission"
impl = importlib.import_module(MODULE)
from engineeringlab import checks
print("Implement submission.py, reload it, then enable exercise checks. Default execution is not a pass.")

Implement submission.py, reload it, then enable exercise checks. Default execution is not a pass.


## Goal
Build and operate your own application image with Docker Compose, implement a release gate and an additive database migration, and rehearse a rollback. Prerequisite: completed Lab 18. Plan two sessions (3–4 hours, an unpiloted estimate). Cloud deployment is optional; local Compose is the required target.

An image packages application code and dependencies. A container is a running instance. The named volume preserves the ticket ledger when the process or container is replaced. A readiness response shows that a service can answer that probe; it does not establish business correctness. Release checks must include ticket creation, approval, tenant isolation, and replay.

## Steps
### 1. Inspect and build your application
Read the Dockerfile, `.dockerignore`, and compose.yaml. Explain each COPY boundary and why instructor solutions and credentials must stay outside the build context. From this folder, after completing Lab 18:

```bash
COURSE_RELEASE=v1 docker compose build
COURSE_RELEASE=v1 docker compose up -d --wait
curl --fail http://127.0.0.1:58018/health
```

Use a second terminal to submit an approval with alpha-reviewer-demo and then the matching ticket with alpha-demo. Use the same JSON operation_id/resource/summary for both requests. Repeat the ticket request and compare IDs. Do not add an approved field: the server forbids it. Save image IDs and responses in student-results; the default starter cannot pass business tests until you implement it.

### 2. TODO 19.1 — make a release decision
Implement `release_gate` in submission.py. Accept exactly tests, failures, unauthorized_effects, replay_duplicates. Counts must be nonnegative integers excluding booleans; at least one test must run, and all three failure counts must be zero. Missing, malformed, or extra fields reject the release. These are teaching criteria, not a complete production risk policy. A hand-written all-zero report is not evidence: retain test output and trace IDs supporting each count.

In [2]:
if RUN_EXERCISES:
    importlib.reload(impl)
    checks.releases(impl)
    print("Release decision contracts passed")

### 3. TODO 19.2 — evolve the ledger
Implement `upgrade_schema(conn)` in submission.py. Use an explicit transaction, inspect PRAGMA table_info(tickets), add created_by TEXT NOT NULL DEFAULT 'legacy' only if absent, and insert schema version 2 only if absent. Preserve rows and support a repeated migration. Roll back if any step fails. Do not drop tables or delete the volume.

Run the migration against the container's ledger using `docker compose exec api python` and your function. Record before/after schema versions, then repeat a pre-migration ticket request. The additive column with a default is chosen to let the previous application keep working; this compatibility must still be tested.

In [3]:
if RUN_EXERCISES:
    importlib.reload(impl)
    checks.migration(impl)
    print("Additive migration, repeat migration, old replay, and new insert passed")

### 4. Exercise a broken release and rollback
Keep your v1 image. Make a controlled application change in your own branch that breaks a contract, run the tests, and confirm the gate rejects it. Repair the change, build v2, run business smoke checks, then roll back with:

```bash
COURSE_RELEASE=v1 docker compose up -d --no-build --wait
```

Verify the reported version, existing ticket IDs, and approval behavior. Rolling back code does not undo database effects. Record whether your schema remains compatible. Stop your course stack with `docker compose down`; retain its named volume. Never use `down -v` as routine cleanup.

## Checks and submission
Submit v1/v2 image IDs, build commands, real API smoke results, migration evidence, gate decisions, and rollback results. Add a test where a dependency is unavailable. Keep that separate from an injected exception test. `ci-template.yml` is an optional GitHub Actions starting point to copy into your own repository; hosted CI is not run by this notebook.

Rubric: working image/Compose 30, gate 20, migration 25, rollback and evidence 25. Automated reference checks do not substitute for your own Docker demonstration.

## Next Steps
Lab 20 measures the service. References: [Docker Compose startup order](https://docs.docker.com/compose/how-tos/startup-order/), [SQLite ALTER TABLE](https://www.sqlite.org/lang_altertable.html), [GitHub Python CI](https://docs.github.com/en/actions/tutorials/build-and-test-code/python).